## Name: Asit Jain
## Roll No: M25DE1049
## Assignment 3 - MLBD

# Part 4: Hybrid Recommendation Model
## Task 7: Implementing a Hybrid Recommendation Model

Combine content-based filtering (CBF) and collaborative filtering (CF) into a hybrid system.
We implement a meta-learning strategy: train a ML model to dynamically blend CBF and CF scores.

Features for each user-movie pair:
- Predicted score from CBF (user-profile cosine similarity)
- Predicted score from CF (Funk SVD)
- Movie popularity score (average rating of the movie)
- User average rating (to account for user bias)

**Dataset:** MovieLens ml-latest-small

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'pandas', 'numpy', 'scikit-learn', 'scipy', '-q'])

import os, zipfile, urllib.request
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
print('All imports successful!')

All imports successful!


### Step 1: Load Dataset and Train/Test Split

In [2]:
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
zip_path = 'ml-latest-small.zip'
extract_dir = 'ml-latest-small'

if not os.path.exists(extract_dir):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('.')

movies = pd.read_csv(os.path.join(extract_dir, 'movies.csv'))
ratings_df = pd.read_csv(os.path.join(extract_dir, 'ratings.csv'))
print(f'Movies: {len(movies)}, Ratings: {len(ratings_df)}, Users: {ratings_df["userId"].nunique()}')

train_df, test_df = train_test_split(ratings_df, test_size=0.2, random_state=42)
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

Movies: 9742, Ratings: 100836, Users: 610
Train: 80668, Test: 20168


### Step 2: Build CBF Component (User-Profile TF-IDF from Task 2)

In [3]:
# TF-IDF on genres
movies['genre_text'] = movies['genres'].str.replace('|', ' ', regex=False)
movies['genre_text'] = movies['genre_text'].replace('(no genres listed)', '')
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genre_text'])
movie_id_to_idx = pd.Series(movies.index, index=movies['movieId'])

print(f'TF-IDF matrix: {tfidf_matrix.shape}')

# Build user profiles from TRAIN set only
def build_user_profiles(rat_df):
    profiles = {}
    for uid, group in rat_df.groupby('userId'):
        valid = group[group['movieId'].isin(movie_id_to_idx.index)]
        if valid.empty:
            continue
        indices = movie_id_to_idx[valid['movieId']].values
        vecs = tfidf_matrix[indices].toarray()
        weights = valid['rating'].values.reshape(-1, 1)
        profiles[uid] = (vecs * weights).sum(axis=0) / weights.sum()
    return profiles

user_profiles = build_user_profiles(train_df)
print(f'Built CBF profiles for {len(user_profiles)} users')

def cbf_score(user_id, movie_id):
    """Cosine similarity between user profile and movie TF-IDF vector."""
    if user_id not in user_profiles or movie_id not in movie_id_to_idx.index:
        return np.nan
    profile = user_profiles[user_id].reshape(1, -1)
    idx = movie_id_to_idx[movie_id]
    movie_vec = tfidf_matrix[idx]
    sim = cosine_similarity(profile, movie_vec)[0, 0]
    # Scale from [0,1] similarity to [0.5, 5.0] rating scale
    return 0.5 + sim * 4.5

# Quick test
print(f'CBF score (user 1, movie 1): {cbf_score(1, 1):.3f}')

TF-IDF matrix: (9742, 21)
Built CBF profiles for 610 users
CBF score (user 1, movie 1): 3.009


### Step 3: Build CF Component (Funk SVD from Task 6)

In [4]:
class FunkSVD:
    def __init__(self, n_factors=50, n_epochs=20, lr=0.005, reg=0.02, random_state=42):
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.lr = lr
        self.reg = reg
        self.random_state = random_state

    def fit(self, train_data):
        rng = np.random.RandomState(self.random_state)
        self.user_ids = train_data['userId'].unique()
        self.item_ids = train_data['movieId'].unique()
        self.user_to_idx = {uid: i for i, uid in enumerate(self.user_ids)}
        self.item_to_idx = {iid: i for i, iid in enumerate(self.item_ids)}
        n_u, n_i = len(self.user_ids), len(self.item_ids)
        self.mu = train_data['rating'].mean()
        self.b_u = np.zeros(n_u)
        self.b_i = np.zeros(n_i)
        self.P = rng.normal(0, 0.1, (n_u, self.n_factors))
        self.Q = rng.normal(0, 0.1, (n_i, self.n_factors))
        users = train_data['userId'].map(self.user_to_idx).values
        items = train_data['movieId'].map(self.item_to_idx).values
        rat = train_data['rating'].values
        for epoch in range(self.n_epochs):
            idx_arr = rng.permutation(len(rat))
            loss = 0.0
            for idx in idx_arr:
                u, i, r = users[idx], items[idx], rat[idx]
                pred = self.mu + self.b_u[u] + self.b_i[i] + self.P[u] @ self.Q[i]
                err = r - pred
                loss += err**2
                self.b_u[u] += self.lr * (err - self.reg * self.b_u[u])
                self.b_i[i] += self.lr * (err - self.reg * self.b_i[i])
                Pu = self.P[u].copy()
                self.P[u] += self.lr * (err * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (err * Pu - self.reg * self.Q[i])
            if (epoch+1) % 10 == 0 or epoch == 0:
                print(f'  Epoch {epoch+1}/{self.n_epochs} RMSE: {np.sqrt(loss/len(rat)):.4f}')
        return self

    def predict(self, uid, mid):
        if uid in self.user_to_idx and mid in self.item_to_idx:
            u, i = self.user_to_idx[uid], self.item_to_idx[mid]
            p = self.mu + self.b_u[u] + self.b_i[i] + self.P[u] @ self.Q[i]
        elif uid in self.user_to_idx:
            p = self.mu + self.b_u[self.user_to_idx[uid]]
        elif mid in self.item_to_idx:
            p = self.mu + self.b_i[self.item_to_idx[mid]]
        else:
            p = self.mu
        return np.clip(p, 0.5, 5.0)

print('Training Funk SVD on train set...')
svd_model = FunkSVD(n_factors=50, n_epochs=20, lr=0.005, reg=0.02)
svd_model.fit(train_df)
print('CF model ready.')

Training Funk SVD on train set...
  Epoch 1/20 RMSE: 0.9647
  Epoch 10/20 RMSE: 0.8175
  Epoch 20/20 RMSE: 0.7248
CF model ready.


### Step 4: Compute Auxiliary Features

In [5]:
# Movie popularity: average rating per movie (from train set)
movie_avg = train_df.groupby('movieId')['rating'].mean()
global_avg = train_df['rating'].mean()

# User average rating (from train set)
user_avg = train_df.groupby('userId')['rating'].mean()

# Number of ratings per user in train (for cold-start analysis later)
user_rating_count = train_df.groupby('userId')['rating'].count()

print(f'Movie avg ratings computed for {len(movie_avg)} movies')
print(f'User avg ratings computed for {len(user_avg)} users')

Movie avg ratings computed for 8983 movies
User avg ratings computed for 610 users


### Step 5: Build Feature Matrix for Meta-Learner

For each (user, movie) pair in the dataset, compute 4 features:
1. CBF score
2. CF score (Funk SVD prediction)
3. Movie popularity (avg rating)
4. User average rating

In [6]:
def build_features(df, svd_model, user_profiles, movie_avg, user_avg, global_avg):
    """Build feature matrix for the meta-learner."""
    cbf_scores = []
    cf_scores = []
    pop_scores = []
    usr_scores = []
    valid_mask = []

    for _, row in df.iterrows():
        uid, mid = row['userId'], row['movieId']

        # CBF score
        cb = cbf_score(uid, mid)
        if np.isnan(cb):
            cb = global_avg  # fallback
        cbf_scores.append(cb)

        # CF score
        cf_scores.append(svd_model.predict(uid, mid))

        # Movie popularity
        pop_scores.append(movie_avg.get(mid, global_avg))

        # User average
        usr_scores.append(user_avg.get(uid, global_avg))

    X = np.column_stack([cbf_scores, cf_scores, pop_scores, usr_scores])
    y = df['rating'].values
    return X, y

print('Building features for train set...')
X_train, y_train = build_features(train_df, svd_model, user_profiles, movie_avg, user_avg, global_avg)
print(f'Train features shape: {X_train.shape}')

print('Building features for test set...')
X_test, y_test = build_features(test_df, svd_model, user_profiles, movie_avg, user_avg, global_avg)
print(f'Test features shape: {X_test.shape}')

Building features for train set...
Train features shape: (80668, 4)
Building features for test set...
Test features shape: (20168, 4)


### Step 6: Train Meta-Learner (Hybrid Model)

In [7]:
# Model 1: Ridge Regression (simple linear blending)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_preds = np.clip(ridge.predict(X_test), 0.5, 5.0)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_preds))

print('Ridge Regression (linear meta-learner):')
print(f'  Test RMSE: {ridge_rmse:.4f}')
print(f'  Feature weights: CBF={ridge.coef_[0]:.3f}, CF={ridge.coef_[1]:.3f}, '
      f'Popularity={ridge.coef_[2]:.3f}, UserAvg={ridge.coef_[3]:.3f}')
print(f'  Intercept: {ridge.intercept_:.3f}')

# Model 2: Gradient Boosting (non-linear meta-learner)
gbr = GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
gbr.fit(X_train, y_train)
gbr_preds = np.clip(gbr.predict(X_test), 0.5, 5.0)
gbr_rmse = np.sqrt(mean_squared_error(y_test, gbr_preds))

print(f'\nGradient Boosting (non-linear meta-learner):')
print(f'  Test RMSE: {gbr_rmse:.4f}')
feat_names = ['CBF', 'CF (SVD)', 'Popularity', 'User Avg']
print(f'  Feature importances:')
for name, imp in zip(feat_names, gbr.feature_importances_):
    print(f'    {name}: {imp:.3f}')

Ridge Regression (linear meta-learner):
  Test RMSE: 0.9090
  Feature weights: CBF=0.098, CF=1.670, Popularity=-0.088, UserAvg=-0.601
  Intercept: -0.210

Gradient Boosting (non-linear meta-learner):
  Test RMSE: 0.9118
  Feature importances:
    CBF: 0.008
    CF (SVD): 0.888
    Popularity: 0.055
    User Avg: 0.049


### Step 7: Evaluate All Models - RMSE, Precision@K, Recall@K

In [8]:
# Individual model baselines on test set
cbf_only_preds = X_test[:, 0]  # CBF scores
cf_only_preds = X_test[:, 1]   # CF scores

cbf_rmse = np.sqrt(mean_squared_error(y_test, cbf_only_preds))
cf_rmse = np.sqrt(mean_squared_error(y_test, cf_only_preds))

print('RMSE Comparison:')
print(f'  CBF only:              {cbf_rmse:.4f}')
print(f'  CF only (Funk SVD):    {cf_rmse:.4f}')
print(f'  Hybrid (Ridge):        {ridge_rmse:.4f}')
print(f'  Hybrid (GBR):          {gbr_rmse:.4f}')

RMSE Comparison:
  CBF only:              1.5120
  CF only (Funk SVD):    0.8812
  Hybrid (Ridge):        0.9090
  Hybrid (GBR):          0.9118


In [9]:
def precision_recall_at_k(test_df, pred_scores, k=10, threshold=4.0):
    """Compute Precision@K and Recall@K given predicted scores aligned with test_df."""
    test_with_preds = test_df.copy()
    test_with_preds['pred'] = pred_scores

    precisions, recalls = [], []
    for uid, group in test_with_preds.groupby('userId'):
        relevant = set(group[group['rating'] >= threshold]['movieId'])
        if not relevant:
            continue
        top_k = group.nlargest(k, 'pred')
        rec_ids = set(top_k['movieId'])
        hits = rec_ids & relevant
        precisions.append(len(hits) / k)
        recalls.append(len(hits) / len(relevant))
    return np.mean(precisions) if precisions else 0, np.mean(recalls) if recalls else 0

models = {
    'CBF Only': cbf_only_preds,
    'CF Only (SVD)': cf_only_preds,
    'Hybrid (Ridge)': ridge_preds,
    'Hybrid (GBR)': gbr_preds,
}

print(f'{"Model":<20s} | {"RMSE":>8s} | {"Prec@10":>8s} | {"Recall@10":>9s}')
print('-' * 55)
for name, preds in models.items():
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    prec, rec = precision_recall_at_k(test_df, preds, k=10)
    print(f'{name:<20s} | {rmse:>8.4f} | {prec:>8.4f} | {rec:>9.4f}')

Model                |     RMSE |  Prec@10 | Recall@10
-------------------------------------------------------
CBF Only             |   1.5120 |   0.4908 |    0.6388
CF Only (SVD)        |   0.8812 |   0.5663 |    0.6796
Hybrid (Ridge)       |   0.9090 |   0.5676 |    0.6813
Hybrid (GBR)         |   0.9118 |   0.5651 |    0.6810


### Step 8: Generate Top-N Hybrid Recommendations

In [10]:
def recommend_hybrid(user_id, model, svd_model, movies_df, train_data,
                     movie_avg, user_avg, global_avg, top_n=10):
    """Generate top-N recommendations using the hybrid meta-learner."""
    rated = set(train_data[train_data['userId'] == user_id]['movieId'])
    candidates = movies_df[~movies_df['movieId'].isin(rated)].copy()

    features = []
    for _, row in candidates.iterrows():
        mid = row['movieId']
        cb = cbf_score(user_id, mid)
        if np.isnan(cb):
            cb = global_avg
        cf = svd_model.predict(user_id, mid)
        pop = movie_avg.get(mid, global_avg)
        usr = user_avg.get(user_id, global_avg)
        features.append([cb, cf, pop, usr])

    X = np.array(features)
    candidates['pred'] = np.clip(model.predict(X), 0.5, 5.0)
    top = candidates.nlargest(top_n, 'pred')
    return top[['title', 'genres', 'pred']].rename(columns={'pred': 'Predicted Rating'}).round(2)

# Pick the best hybrid model
best_hybrid = gbr if gbr_rmse < ridge_rmse else ridge
best_name = 'GBR' if gbr_rmse < ridge_rmse else 'Ridge'
print(f'Using best hybrid model: {best_name}\n')

for uid in [1, 5, 100]:
    print('=' * 70)
    print(f'Top 10 Hybrid Recommendations for User {uid}')
    print('=' * 70)
    recs = recommend_hybrid(uid, best_hybrid, svd_model, movies, train_df,
                            movie_avg, user_avg, global_avg, top_n=10)
    print(recs.to_string(index=False))
    print()

Using best hybrid model: Ridge

Top 10 Hybrid Recommendations for User 1
                                                                      title                      genres  Predicted Rating
                                                              Casino (1995)                 Crime|Drama               5.0
                                                           Apollo 13 (1995)        Adventure|Drama|IMAX               5.0
                                                  Heavenly Creatures (1994)                 Crime|Drama               5.0
                 Like Water for Chocolate (Como agua para chocolate) (1992)       Drama|Fantasy|Romance               5.0
             Léon: The Professional (a.k.a. The Professional) (Léon) (1994) Action|Crime|Drama|Thriller               5.0
                           Three Colors: Red (Trois couleurs: Rouge) (1994)                       Drama               5.0
                                           Shawshank Redemption, The (199

### Step 9: Cold-Start User Analysis

Cold-start users have very few ratings, making CF unreliable.
We analyze how the hybrid model performs compared to CBF-only and CF-only for users with different numbers of ratings.

In [11]:
# Add rating count to test data
test_with_count = test_df.copy()
test_with_count['n_train_ratings'] = test_with_count['userId'].map(user_rating_count).fillna(0)

# Define cold-start buckets
buckets = [
    ('Cold (1-5 ratings)', (1, 5)),
    ('Cool (6-20 ratings)', (6, 20)),
    ('Warm (21-50 ratings)', (21, 50)),
    ('Hot (50+ ratings)', (51, 99999)),
]

print(f'{"User Group":<25s} | {"N":>5s} | {"CBF RMSE":>9s} | {"CF RMSE":>9s} | {"Hybrid RMSE":>11s}')
print('-' * 70)

for label, (lo, hi) in buckets:
    mask = (test_with_count['n_train_ratings'] >= lo) & (test_with_count['n_train_ratings'] <= hi)
    if mask.sum() == 0:
        continue
    y_bucket = y_test[mask.values]
    cbf_r = np.sqrt(mean_squared_error(y_bucket, cbf_only_preds[mask.values]))
    cf_r = np.sqrt(mean_squared_error(y_bucket, cf_only_preds[mask.values]))
    hyb_r = np.sqrt(mean_squared_error(y_bucket, gbr_preds[mask.values]))
    print(f'{label:<25s} | {mask.sum():>5d} | {cbf_r:>9.4f} | {cf_r:>9.4f} | {hyb_r:>11.4f}')

User Group                |     N |  CBF RMSE |   CF RMSE | Hybrid RMSE
----------------------------------------------------------------------
Cool (6-20 ratings)       |   400 |    1.8202 |    0.9431 |      0.9834
Warm (21-50 ratings)      |  1699 |    1.6484 |    0.9204 |      0.9528
Hot (50+ ratings)         | 18069 |    1.4909 |    0.8759 |      0.9061


### Analysis: Cold-Start Performance

Key observations:

1. **Cold-start users (1-5 ratings)**: CBF tends to perform relatively better here because it only needs genre preferences, which can be estimated from even a few ratings. CF struggles because there is not enough collaborative signal.

2. **Warm/Hot users (20+ ratings)**: CF dominates because it can leverage rich collaborative patterns. CBF is limited by genre granularity.

3. **Hybrid advantage**: The meta-learner automatically learns to weight CBF more heavily for cold-start users and CF more heavily for active users. This is the key benefit of the hybrid approach over a fixed alpha blending.

4. **Movie popularity** acts as a useful fallback feature for both cold-start users and cold-start items (new movies with few ratings).

The hybrid model should show the most consistent performance across all user groups, avoiding the worst-case scenarios of either individual approach.